# Tutorial 09: HLLSet De Bruijn — Evolution Recovery with D/R/N Transformations

This tutorial introduces the `hllset_debruijn` module, which applies De Bruijn graph concepts to HLLSet evolution sequences.

## What You'll Learn

1. **D/R/N Transformations** — Deleted, Retained, Novel decomposition
2. **HLLSet De Bruijn Graphs** — Nodes are states, edges are transformations
3. **Evolution Path Recovery** — Ordering unordered HLLSet collections
4. **Transition Verification** — Ensuring algebraic consistency

## The Key Insight

For any transition from HLLSet H₁ to H₂:

$$H_2 = (H_1 \setminus D) \cup R \cup N$$

Where:
- **D (Deleted)** = H₁ \ H₂ — tokens removed
- **R (Retained)** = H₁ ∩ H₂ — tokens preserved
- **N (Novel)** = H₂ \ H₁ — tokens added

This decomposition is **fully invertible** — given (D, R, N), we can reconstruct the transition.

In [1]:
# Import modules
import sys
sys.path.insert(0, '..')

from core.hllset_debruijn import (
    DRNTriple,
    FullDRNTriple,
    HLLSetEdge,
    HLLSetDeBruijnGraph,
    decompose_transition,
    full_decompose_transition,
    verify_reconstruction,
    build_hllset_debruijn,
    find_evolution_path,
)
from core.hllset import HLLSet
from core.bss import bss

## 1. The D/R/N Triple

The `DRNTriple` captures the complete transformation between two HLLSets.

In [2]:
# Create two HLLSets representing states
H1 = HLLSet.from_batch(["apple", "banana", "cherry", "date"])
H2 = HLLSet.from_batch(["banana", "cherry", "elderberry", "fig"])

print(f"H₁: ['apple', 'banana', 'cherry', 'date']")
print(f"H₂: ['banana', 'cherry', 'elderberry', 'fig']")
print(f"\n|H₁| ≈ {H1.cardinality():.0f}")
print(f"|H₂| ≈ {H2.cardinality():.0f}")

H₁: ['apple', 'banana', 'cherry', 'date']
H₂: ['banana', 'cherry', 'elderberry', 'fig']

|H₁| ≈ 5
|H₂| ≈ 6


In [3]:
# Compute D/R/N decomposition
drn = decompose_transition(H1, H2)

print(f"D/R/N Decomposition:")
print(f"  D (deleted):  |D| ≈ {drn.deleted_card:.0f}  (apple, date)")
print(f"  R (retained): |R| ≈ {drn.retained_card:.0f}  (banana, cherry)")
print(f"  N (novel):    |N| ≈ {drn.novel_card:.0f}  (elderberry, fig)")

D/R/N Decomposition:
  D (deleted):  |D| ≈ 3  (apple, date)
  R (retained): |R| ≈ 3  (banana, cherry)
  N (novel):    |N| ≈ 4  (elderberry, fig)


In [4]:
# DRNTriple properties
print(f"Transition properties:")
print(f"  is_growth: {drn.is_growth()}  (more added than deleted)")
print(f"  is_decay:  {drn.is_decay()}  (more deleted than added)")
print(f"  net_change: {drn.net_change():.0f}  (|N| - |D|)")

Transition properties:
  is_growth: True  (more added than deleted)
  is_decay:  False  (more deleted than added)
  net_change: 1  (|N| - |D|)


## 2. Verifying the Reconstruction

The fundamental invariant: H₂ ≈ (H₁ \ D) ∪ R ∪ N

In [5]:
# Verify algebraically
step1 = H1.diff(drn.deleted)        # H₁ \ D
step2 = step1.union(drn.retained)   # ... ∪ R
reconstructed = step2.union(drn.novel)  # ... ∪ N

print(f"Reconstruction verification:")
print(f"  |H₂|           ≈ {H2.cardinality():.1f}")
print(f"  |reconstructed| ≈ {reconstructed.cardinality():.1f}")
print(f"  Difference: {abs(H2.cardinality() - reconstructed.cardinality()):.2f}")

Reconstruction verification:
  |H₂|           ≈ 6.0
  |reconstructed| ≈ 6.0
  Difference: 0.00


In [6]:
# Use the verification function
is_valid = verify_reconstruction(H1, H2, drn, tolerance=2.0)
print(f"Reconstruction valid: {is_valid}")

Reconstruction valid: True


## 3. Full D/R/N with Token Recovery

The `FullDRNTriple` preserves original token ORDER for each component.

In [7]:
# Full decomposition with token lists
tokens1 = ["apple", "banana", "cherry", "date"]
tokens2 = ["banana", "cherry", "elderberry", "fig"]

full_drn = full_decompose_transition(tokens1, tokens2)

print(f"Full D/R/N with tokens:")
print(f"  Deleted tokens:  {full_drn.deleted_tokens}")
print(f"  Retained tokens: {full_drn.retained_tokens}")
print(f"  Novel tokens:    {full_drn.novel_tokens}")

Full D/R/N with tokens:
  Deleted tokens:  ('apple', 'date')
  Retained tokens: ('banana', 'cherry')
  Novel tokens:    ('elderberry', 'fig')


In [8]:
# Each component is also an HLLSet
print(f"\nHLLSet cardinalities:")
print(f"  |D_hll| ≈ {full_drn.deleted_hll.cardinality():.0f}")
print(f"  |R_hll| ≈ {full_drn.retained_hll.cardinality():.0f}")
print(f"  |N_hll| ≈ {full_drn.novel_hll.cardinality():.0f}")

# Get basic DRN triple
basic_drn = full_drn.drn
print(f"\nBasic DRN: {basic_drn}")


HLLSet cardinalities:
  |D_hll| ≈ 3
  |R_hll| ≈ 3
  |N_hll| ≈ 4

Basic DRN: DRNTriple(deleted=HLLSet(3bf868ed..., |A|≈3.0, backend=C/Cython), retained=HLLSet(ef34c2bc..., |A|≈3.0, backend=C/Cython), novel=HLLSet(d5a98a60..., |A|≈4.0, backend=C/Cython))


## 4. HLLSet De Bruijn Graph

Build a De Bruijn-like graph where:
- **Nodes** = HLLSet states
- **Edges** = (D, R, N) transformations where BSS meets thresholds

In [9]:
# Create a sequence of evolving HLLSets
states = [
    HLLSet.from_batch(["a", "b", "c"]),           # State 0
    HLLSet.from_batch(["b", "c", "d"]),           # State 1: -a, +d
    HLLSet.from_batch(["c", "d", "e"]),           # State 2: -b, +e
    HLLSet.from_batch(["d", "e", "f"]),           # State 3: -c, +f
]

labels = ["S0", "S1", "S2", "S3"]

print(f"Evolution sequence:")
for i, (state, label) in enumerate(zip(states, labels)):
    print(f"  {label}: |H| ≈ {state.cardinality():.0f}")

Evolution sequence:
  S0: |H| ≈ 4
  S1: |H| ≈ 4
  S2: |H| ≈ 5
  S3: |H| ≈ 5


In [10]:
# Build HLLSet De Bruijn graph
graph = build_hllset_debruijn(
    states,
    labels=labels,
    tau_min=0.3,   # Minimum overlap required
    rho_max=0.7,   # Maximum exclusion allowed
)

print(f"HLLSet De Bruijn Graph:")
print(f"  Nodes: {graph.num_nodes}")
print(f"  Edges: {graph.num_edges}")

HLLSet De Bruijn Graph:
  Nodes: 4
  Edges: 8


In [11]:
# Examine edges (transitions)
print(f"\nEdges (transitions):")
for edge in graph.edges:
    src = labels[edge.source_idx]
    tgt = labels[edge.target_idx]
    print(f"  {src} → {tgt}:")
    print(f"    BSS: τ={edge.tau:.3f}, ρ={edge.rho:.3f}")
    print(f"    D/R/N: |D|≈{edge.drn.deleted_card:.0f}, |R|≈{edge.drn.retained_card:.0f}, |N|≈{edge.drn.novel_card:.0f}")


Edges (transitions):
  S0 → S1:
    BSS: τ=0.750, ρ=0.500
    D/R/N: |D|≈2, |R|≈3, |N|≈2
  S0 → S2:
    BSS: τ=0.400, ρ=0.400
    D/R/N: |D|≈2, |R|≈2, |N|≈3
  S1 → S0:
    BSS: τ=0.750, ρ=0.500
    D/R/N: |D|≈2, |R|≈3, |N|≈2
  S1 → S2:
    BSS: τ=0.800, ρ=0.400
    D/R/N: |D|≈2, |R|≈4, |N|≈2
  S1 → S3:
    BSS: τ=0.400, ρ=0.600
    D/R/N: |D|≈3, |R|≈2, |N|≈3
  S2 → S1:
    BSS: τ=1.000, ρ=0.500
    D/R/N: |D|≈2, |R|≈4, |N|≈2
  S2 → S3:
    BSS: τ=0.600, ρ=0.400
    D/R/N: |D|≈2, |R|≈3, |N|≈2
  S3 → S2:
    BSS: τ=0.600, ρ=0.400
    D/R/N: |D|≈2, |R|≈3, |N|≈2


## 5. Evolution Path Finding

Given an unordered collection of HLLSets, find the most likely evolution order.

In [12]:
# Shuffle the states (simulate unordered input)
import random
shuffled_states = states.copy()
shuffled_labels = labels.copy()

# Shuffle together
combined = list(zip(shuffled_states, shuffled_labels))
random.shuffle(combined)
shuffled_states, shuffled_labels = zip(*combined)
shuffled_states = list(shuffled_states)
shuffled_labels = list(shuffled_labels)

print(f"Shuffled order: {shuffled_labels}")

Shuffled order: ['S1', 'S3', 'S2', 'S0']


In [13]:
# Build graph from shuffled states
shuffled_graph = build_hllset_debruijn(
    shuffled_states,
    labels=shuffled_labels,
    tau_min=0.3,
    rho_max=0.7,
)

# Find evolution path
path = find_evolution_path(shuffled_graph)

print(f"Recovered evolution order:")
recovered_order = [shuffled_labels[i] for i in path]
print(f"  {' → '.join(recovered_order)}")
print(f"\nOriginal order: S0 → S1 → S2 → S3")

Recovered evolution order:
  S1 → S2 → S3

Original order: S0 → S1 → S2 → S3


## 6. Node Degree Analysis

Analyze the graph structure to understand connectivity.

In [14]:
# Degree analysis
print(f"Node degree analysis:")
for i in range(graph.num_nodes):
    out_deg = graph.out_degree(i)
    in_deg = graph.in_degree(i)
    print(f"  {labels[i]}: in={in_deg}, out={out_deg}")

Node degree analysis:
  S0: in=1, out=2
  S1: in=2, out=3
  S2: in=3, out=2
  S3: in=2, out=1


In [15]:
# Adjacency list
adj = graph.adjacency_list()

print(f"\nAdjacency list:")
for node_idx, neighbors in adj.items():
    if neighbors:
        neighbor_str = ", ".join(f"{labels[n]}(τ={t:.2f})" for n, t in neighbors)
        print(f"  {labels[node_idx]} → [{neighbor_str}]")
    else:
        print(f"  {labels[node_idx]} → []")


Adjacency list:
  S0 → [S1(τ=0.75), S2(τ=0.40)]
  S1 → [S0(τ=0.75), S2(τ=0.80), S3(τ=0.40)]
  S2 → [S1(τ=1.00), S3(τ=0.60)]
  S3 → [S2(τ=0.60)]


## 7. DOT Visualization

Generate DOT format for graph visualization.

In [16]:
# Generate DOT representation
dot = graph.to_dot(title="Evolution")
print(dot)

digraph Evolution {
  rankdir=LR;
  node [shape=ellipse];
  H0 [label="S0\n|H|≈4"];
  H1 [label="S1\n|H|≈4"];
  H2 [label="S2\n|H|≈5"];
  H3 [label="S3\n|H|≈5"];
  H0 -> H1 [label="τ=0.75\nD:2 R:3 N:2"];
  H0 -> H2 [label="τ=0.40\nD:2 R:2 N:3"];
  H1 -> H0 [label="τ=0.75\nD:2 R:3 N:2"];
  H1 -> H2 [label="τ=0.80\nD:2 R:4 N:2"];
  H1 -> H3 [label="τ=0.40\nD:3 R:2 N:3"];
  H2 -> H1 [label="τ=1.00\nD:2 R:4 N:2"];
  H2 -> H3 [label="τ=0.60\nD:2 R:3 N:2"];
  H3 -> H2 [label="τ=0.60\nD:2 R:3 N:2"];
}


## 8. Transition Classification

Classify transitions by their D/R/N characteristics.

In [17]:
# Classify transitions
print(f"Transition classification:")
for edge in graph.edges:
    src = labels[edge.source_idx]
    tgt = labels[edge.target_idx]
    drn = edge.drn
    
    if drn.is_growth():
        kind = "GROWTH"
    elif drn.is_decay():
        kind = "DECAY"
    else:
        kind = "STABLE"
    
    print(f"  {src} → {tgt}: {kind} (net: {drn.net_change():+.0f})")

Transition classification:
  S0 → S1: STABLE (net: +0)
  S0 → S2: GROWTH (net: +1)
  S1 → S0: STABLE (net: +0)
  S1 → S2: STABLE (net: +0)
  S1 → S3: STABLE (net: +0)
  S2 → S1: STABLE (net: +0)
  S2 → S3: STABLE (net: +0)
  S3 → S2: STABLE (net: +0)


## 9. Practical Example: Document Evolution

Track how a document changes through revisions.

In [18]:
# Document versions (simplified)
v1 = "machine learning is powerful"
v2 = "deep learning is powerful and fast"
v3 = "deep neural networks are powerful fast and accurate"

doc_states = [
    HLLSet.from_batch(v1.split()),
    HLLSet.from_batch(v2.split()),
    HLLSet.from_batch(v3.split()),
]

doc_labels = ["v1", "v2", "v3"]

print(f"Document evolution:")
print(f"  v1: '{v1}'")
print(f"  v2: '{v2}'")
print(f"  v3: '{v3}'")

Document evolution:
  v1: 'machine learning is powerful'
  v2: 'deep learning is powerful and fast'
  v3: 'deep neural networks are powerful fast and accurate'


In [19]:
# Build evolution graph
doc_graph = build_hllset_debruijn(
    doc_states,
    labels=doc_labels,
    tau_min=0.2,
    rho_max=0.8,
)

print(f"\nEvolution graph:")
for edge in doc_graph.edges:
    src = doc_labels[edge.source_idx]
    tgt = doc_labels[edge.target_idx]
    drn = edge.drn
    print(f"  {src} → {tgt}:")
    print(f"    Deleted: ~{drn.deleted_card:.0f} tokens")
    print(f"    Retained: ~{drn.retained_card:.0f} tokens")
    print(f"    Novel: ~{drn.novel_card:.0f} tokens")


Evolution graph:
  v1 → v2:
    Deleted: ~2 tokens
    Retained: ~4 tokens
    Novel: ~4 tokens
  v1 → v3:
    Deleted: ~4 tokens
    Retained: ~2 tokens
    Novel: ~8 tokens
  v2 → v1:
    Deleted: ~4 tokens
    Retained: ~4 tokens
    Novel: ~2 tokens
  v2 → v3:
    Deleted: ~3 tokens
    Retained: ~5 tokens
    Novel: ~4 tokens
  v3 → v2:
    Deleted: ~4 tokens
    Retained: ~5 tokens
    Novel: ~3 tokens


In [20]:
# Detailed token-level analysis with full decomposition
print(f"\nDetailed v1 → v2 transition:")
full_v1_v2 = full_decompose_transition(v1.split(), v2.split())
print(f"  Deleted: {full_v1_v2.deleted_tokens}")
print(f"  Retained: {full_v1_v2.retained_tokens}")
print(f"  Novel: {full_v1_v2.novel_tokens}")


Detailed v1 → v2 transition:
  Deleted: ('machine',)
  Retained: ('learning', 'is', 'powerful')
  Novel: ('deep', 'and', 'fast')


## 10. Edge Queries

Query edges by source or target node.

In [21]:
# Get outgoing edges from a node
out_edges = graph.out_edges(0)  # From S0
print(f"Outgoing edges from {labels[0]}:")
for edge in out_edges:
    print(f"  → {labels[edge.target_idx]}: τ={edge.tau:.3f}")

Outgoing edges from S0:
  → S1: τ=0.750
  → S2: τ=0.400


In [22]:
# Get incoming edges to a node
in_edges = graph.in_edges(3)  # To S3
print(f"Incoming edges to {labels[3]}:")
for edge in in_edges:
    print(f"  ← {labels[edge.source_idx]}: τ={edge.tau:.3f}")

Incoming edges to S3:
  ← S1: τ=0.400
  ← S2: τ=0.600


## Summary

In this tutorial, you learned:

1. **D/R/N Decomposition** — Complete transformation representation
2. **Invertibility** — H₂ = (H₁ \ D) ∪ R ∪ N
3. **HLLSet De Bruijn** — Graph of states connected by transformations
4. **Evolution Recovery** — Find order from unordered collections

### The D/R/N Triple

| Component | Formula | Meaning |
|-----------|---------|--------|
| D (Deleted) | H₁ \ H₂ | Tokens removed |
| R (Retained) | H₁ ∩ H₂ | Tokens preserved |
| N (Novel) | H₂ \ H₁ | Tokens added |

### Key Properties

- **is_growth()**: |N| > |D| — forward evolution
- **is_decay()**: |D| > |N| — backward evolution
- **net_change()**: |N| - |D| — net cardinality change

### Connection to Framework

- **D, R, N are HLLSets** — support all algebraic operations
- **FullDRNTriple** preserves token order for De Bruijn recovery
- **BSS thresholds** determine edge existence

### Next Steps

- **Tutorial 10**: Disambiguation — Recover tokens from HLLSet fingerprints